Здесь используются данные по факторам, которые влияют на цену домов в пригороде Бостона (https://www.kaggle.com/datasets/vikrishnan/boston-house-prices).

In [ ]:
import sqlite3
import pandas as pd

In [ ]:
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/housing/housing.data"

Загружаем базу данных по ссылке

In [ ]:
df = pd.read_csv(url)

In [ ]:
df = pd.read_csv(url, delim_whitespace=True, header=None,
                 names=['CRIM', 'ZN', 'INDUS', 'CHAS', 'NOX', 'RM', 'AGE', 'DIS',
                        'RAD', 'TAX', 'PTRATIO', 'B', 'LSTAT', 'MEDV'])

/tmp/ipython-input-2635334535.py:1: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  df = pd.read_csv(url, delim_whitespace=True, header=None,


Здесь было указано, что данные разделены пробелами, и вручную добавляются названия столбцов


In [ ]:
print(df.describe().round(3))

          CRIM       ZN    INDUS     CHAS      NOX       RM      AGE      DIS  \
count  506.000  506.000  506.000  506.000  506.000  506.000  506.000  506.000   
mean     3.614   11.364   11.137    0.069    0.555    6.285   68.575    3.795   
std      8.602   23.322    6.860    0.254    0.116    0.703   28.149    2.106   
min      0.006    0.000    0.460    0.000    0.385    3.561    2.900    1.130   
25%      0.082    0.000    5.190    0.000    0.449    5.885   45.025    2.100   
50%      0.257    0.000    9.690    0.000    0.538    6.208   77.500    3.207   
75%      3.677   12.500   18.100    0.000    0.624    6.624   94.075    5.188   
max     88.976  100.000   27.740    1.000    0.871    8.780  100.000   12.126   

           RAD      TAX  PTRATIO        B    LSTAT     MEDV  
count  506.000  506.000  506.000  506.000  506.000  506.000  
mean     9.549  408.237   18.456  356.674   12.653   22.533  
std      8.707  168.537    2.165   91.295    7.141    9.197  
min      1.000  187.00

Немного описательных статистик для дальнейшего анализа

In [ ]:
con = sqlite3.connect('boston.db')
df.to_sql('boston_housing', con, if_exists='replace', index=False)

506

Создается подключение к sqlite базе данных и сохраняется датафрейм в sql таблицу

# 1. Выведите количество пустых значений по колонкам CRIM, ZN, INDUS, CHAS, NOX (название колонки, кол-во пустых значений)

In [ ]:
# реализация задачи с использованием SQL
pd.read_sql(
    """
SELECT CRIM as column_name, COUNT(*) - COUNT(CRIM) as null_val
FROM boston_housing
UNION ALL
SELECT ZN as column_name, COUNT(*) - COUNT(ZN) as null_val
FROM boston_housing
UNION ALL
SELECT INDUS as column_name, COUNT(*) - COUNT(INDUS) as null_val
FROM boston_housing
UNION ALL
SELECT CHAS as column_name, COUNT(*) - COUNT(CHAS) as null_val
FROM boston_housing
UNION ALL
SELECT NOX as column_name, COUNT(*) - COUNT(NOX) as null_val
FROM boston_housing
    """,
    con,
)

,column_name,null_val
0,0.00632,0
1,18.00000,0
2,2.31000,0
3,0.00000,0
4,0.53800,0


In [ ]:
# реализация задачи с использованием библиотеки pandas
check = ['CRIM', 'ZN', 'INDUS', 'CHAS', 'NOX']
null_counts = df[check].isnull().sum().reset_index()
null_counts.columns = ['column_name', 'null_count']
print(null_counts)

  column_name  null_count
0        CRIM           0
1          ZN           0
2       INDUS           0
3        CHAS           0
4         NOX           0


Как итог, данные не имеют пропусков. В SQL варианте были подсчитаны сначала все строки, потом строки где нет пропусков и нашли разницу между ними, которая показывает количество строк с NULL. UNION ALL объединяет результаты между запросами в одну таблицу. В Python сначала выбираются нужные колонки с помощью check, далее isnull обнаруживает пустые значения и возращает true или false, которые суммируются (sum) и возращаются в обычный датафрейм (reset_index).

# 2. Выведите количество уникальных значений по колонокам CRIM, ZN, INDUS, CHAS, NOX (название колонки, кол-во уникальных значений)

In [ ]:
# реализация задачи с использованием SQL
pd.read_sql(
    """
SELECT 'CRIM' as column, COUNT(DISTINCT CRIM) as unique_val
FROM boston_housing
UNION ALL
SELECT 'ZN' as column, COUNT(DISTINCT ZN) as unique_val
FROM boston_housing
UNION ALL
SELECT 'INDUS' as column, COUNT(DISTINCT INDUS) as unique_val
FROM boston_housing
UNION ALL
SELECT 'CHAS' as column, COUNT(DISTINCT CHAS) as unique_val
FROM boston_housing
UNION ALL
SELECT 'NOX' as column, COUNT(DISTINCT NOX) as unique_val
FROM boston_housing
ORDER BY 2
    """,
    con,
)

,column,unique_val
0,CHAS,2
1,ZN,26
2,INDUS,76
3,NOX,81
4,CRIM,504


In [ ]:
# реализация задачи с использованием библиотеки pandas
check = ['CRIM', 'ZN', 'INDUS', 'CHAS', 'NOX']
unique_counts = df[check].nunique().reset_index()
unique_counts.columns = ['column_name', 'unique_count']
print(unique_counts)


  column_name  unique_count
0        CRIM           504
1          ZN            26
2       INDUS            76
3        CHAS             2
4         NOX            81


В SQL варианте с помощью distinct подсчитываются уникальные значения, каждый запрос соединяется с помощью UNION ALL, а также используя ORDER BY уникальные значения становятся в порядке возрастания. В pandas выираются нужные колонки и с помощью функции nunique() считаются уникальные значения для каждой колонки. Если говорить про получившиеся результаты, то стоит выделить переменную CHAS: 2 уникальных значения подтверждают, что это бинарная переменная.

# 3. Выведите колонки, у которых медиана равна минимальному значению (название колонки) выбирая из CRIM, ZN, INDUS, CHAS, NOX.
Напишите какой вывод можно сделать по данным в этих колонках

In [ ]:
# реализация задачи с использованием SQL
pd.read_sql(
    """
WITH median AS (
  SELECT 'CRIM' as column,
  (SELECT CRIM
  FROM boston_housing
  ORDER BY CRIM
  LIMIT 1 OFFSET (SELECT COUNT(*) / 2 FROM boston_housing)) as median_value,
  (SELECT MIN(CRIM) FROM boston_housing) as min_value
UNION ALL
  SELECT 'ZN' as column,
  (SELECT ZN
  FROM boston_housing
  ORDER BY ZN
  LIMIT 1 OFFSET (SELECT COUNT(*) / 2 FROM boston_housing)) as median_value,
  (SELECT MIN(ZN) FROM boston_housing) as min_value
UNION ALL
  SELECT 'INDUS' as column,
  (SELECT INDUS
  FROM boston_housing
  ORDER BY INDUS
  LIMIT 1 OFFSET (SELECT COUNT(*) / 2 FROM boston_housing)) as median_value,
  (SELECT MIN(INDUS) FROM boston_housing) as min_value
UNION ALL
  SELECT 'CHAS' as column,
  (SELECT CHAS
  FROM boston_housing
  ORDER BY CHAS
  LIMIT 1 OFFSET (SELECT COUNT(*) / 2 FROM boston_housing)) as median_value,
  (SELECT MIN(CHAS) FROM boston_housing) as min_value
UNION ALL
  SELECT 'NOX' as column,
  (SELECT NOX
  FROM boston_housing
  ORDER BY NOX
  LIMIT 1 OFFSET (SELECT COUNT(*) / 2 FROM boston_housing)) as median_value,
  (SELECT MIN(NOX) FROM boston_housing) as min_value
)
SELECT column
FROM median
WHERE median_value = min_value
    """,
    con,
)

,column
0,ZN
1,CHAS


In [ ]:
# реализация задачи с использованием библиотеки pandas
import numpy as np
check = ['CRIM', 'ZN', 'INDUS', 'CHAS', 'NOX']
result = [col for col in check
                 if np.isclose(df[col].median(), df[col].min())]
median_min_df = pd.DataFrame({'column': result})
print(median_min_df)

  column
0     ZN
1   CHAS


В sql коде используется cte дл создания временной таблицы для каждой колонки, медиана вычисляется через сортировку и offset, который берет серединное значение, и находится минимальное значение через min(). Так делается для каждого выбранного столбца, все соединяется с UNION ALL. В конце выводятся называния колонок из временной таблицы и фильтруются колонки, где медиана равна минимуму. В Python варианте используются встроенные в библиотеку pandas функции для медианы и минимума. затем используется функция из numpy np.isclose для сравнения чисел.
Как итог, по переменной ZN (доля жилой земли, разделенной на участки площадью более 25 000 кв.фунтов) медиана равная 0 означает, что более 50% районов не имеют земли под крупные участки, скорее всего они плотно застроены / городская застройка. По переменной CHAS (бинарная, означающающая граничит ли район с рекой или нет) медиана также равна минимальному значению 0, более половины районов не граничат с рекой.

# 4. Выведите разницу между средним количеством комнат(RM) в домах с самой дорогой стоимостью(MEDV) и 25 самыми дешевыми домами.
Аналогично по 50, 100, 200, 300 самыми дешевыми домами. (кол-во домов(25,50,100,200,300), среднее кол-во комнат в них, среднее кол-во комнат в самых дорогих, разница).
Напишите влияет ли кол-во комнат на стоимость и как сильно.

In [ ]:
# реализация задачи с использованием SQL
pd.read_sql(
    """
SELECT
    25 as house_count,
    (SELECT AVG(RM) FROM (SELECT RM FROM boston_housing ORDER BY MEDV LIMIT 25)) as avg_rm_cheap,
    (SELECT AVG(RM) FROM (SELECT RM FROM boston_housing ORDER BY MEDV DESC LIMIT 25)) as avg_rm_exp,
    (SELECT AVG(RM) FROM (SELECT RM FROM boston_housing ORDER BY MEDV DESC LIMIT 25)) -
    (SELECT AVG(RM) FROM (SELECT RM FROM boston_housing ORDER BY MEDV LIMIT 25)) as difference
UNION ALL
SELECT
    50 as house_count,
    (SELECT AVG(RM) FROM (SELECT RM FROM boston_housing ORDER BY MEDV LIMIT 50)) as avg_rm_cheap,
    (SELECT AVG(RM) FROM (SELECT RM FROM boston_housing ORDER BY MEDV DESC LIMIT 25)) as avg_rm_exp,
    (SELECT AVG(RM) FROM (SELECT RM FROM boston_housing ORDER BY MEDV DESC LIMIT 25)) -
    (SELECT AVG(RM) FROM (SELECT RM FROM boston_housing ORDER BY MEDV LIMIT 50)) as difference
UNION ALL
SELECT
    100 as house_count,
    (SELECT AVG(RM) FROM (SELECT RM FROM boston_housing ORDER BY MEDV LIMIT 100)) as avg_rm_cheap,
    (SELECT AVG(RM) FROM (SELECT RM FROM boston_housing ORDER BY MEDV DESC LIMIT 25)) as avg_rm_exp,
    (SELECT AVG(RM) FROM (SELECT RM FROM boston_housing ORDER BY MEDV DESC LIMIT 25)) -
    (SELECT AVG(RM) FROM (SELECT RM FROM boston_housing ORDER BY MEDV LIMIT 100)) as difference
UNION ALL
SELECT
    200 as house_count,
    (SELECT AVG(RM) FROM (SELECT RM FROM boston_housing ORDER BY MEDV LIMIT 200)) as avg_rm_cheap,
    (SELECT AVG(RM) FROM (SELECT RM FROM boston_housing ORDER BY MEDV DESC LIMIT 25)) as avg_rm_exp,
    (SELECT AVG(RM) FROM (SELECT RM FROM boston_housing ORDER BY MEDV DESC LIMIT 25)) -
    (SELECT AVG(RM) FROM (SELECT RM FROM boston_housing ORDER BY MEDV LIMIT 200)) as difference
UNION ALL
SELECT
    300 as house_count,
    (SELECT AVG(RM) FROM (SELECT RM FROM boston_housing ORDER BY MEDV LIMIT 300)) as avg_rm_cheap,
    (SELECT AVG(RM) FROM (SELECT RM FROM boston_housing ORDER BY MEDV DESC LIMIT 25)) as avg_rm_exp,
    (SELECT AVG(RM) FROM (SELECT RM FROM boston_housing ORDER BY MEDV DESC LIMIT 25)) -
    (SELECT AVG(RM) FROM (SELECT RM FROM boston_housing ORDER BY MEDV LIMIT 300)) as difference
    """,
    con,
)

,house_count,avg_rm_cheap,avg_rm_exp,difference
0,25,5.747840,7.63732,1.889480
1,50,5.753240,7.63732,1.884080
2,100,5.887120,7.63732,1.750200
3,200,5.911705,7.63732,1.725615
4,300,5.972227,7.63732,1.665093


In [ ]:
# реализация задачи с использованием библиотеки pandas
expensive_avg_rm = df.nlargest(25, 'MEDV')['RM'].mean()
sizes = [25, 50, 100, 200, 300]

result_df = pd.DataFrame([{
    'house_count': size,
    'avg_rm_cheap': df.nsmallest(size, 'MEDV')['RM'].mean(),
    'avg_rm_exp': expensive_avg_rm,
    'difference': expensive_avg_rm - df.nsmallest(size, 'MEDV')['RM'].mean()
} for size in sizes])
print(result_df)

correlation = df['RM'].corr(df['MEDV'])
print(f'Корреляция между RV-MEDV: {correlation.round(3)}')


   house_count  avg_rm_cheap  avg_rm_exp  difference
0           25      5.747840     7.63732    1.889480
1           50      5.753240     7.63732    1.884080
2          100      5.887120     7.63732    1.750200
3          200      5.911705     7.63732    1.725615
4          300      5.972227     7.63732    1.665093
Корреляция между RV-MEDV: 0.695


В SQL запросе через сортировку находятся 25 самых дорогих домов и 25/50/100 и тд самых дешевых, подзапросы вычисляют среднее количество комнат для каждой группы и находится разница между самыми дорогими и дешевыми. В варианте python также выбираются 25 самых дорогих домов и вычисляется их среднее значение и создается список с размерами выборки данных. Для каждого размера выбираются самые дешевые дома и вычисляется среднее по ним, а дальше вычисляется разница.
Как итог, дорогие дома имеют большее количество комнат, что достаточно логично. Разница уменьшается по мере добавления большего количества "дешевых" домов, так как это уже половина и больше выборки, а значит дома могут стоить уже ближе к дорогим домам. Чтобы понять влияние, можно сделать матрицу корреляций между количеством комнат и стоимостью - значение составляет 0,695, что говорит о сильной связи между переменными: чем больше количество комнат, тем дороже дом.

# 5. Выведите ранги значений колонки LSTAT(процент населения с более низким статусом) в домах с самой дорогой стоимостью (значение LSTAT, стоимость, ранг). Напишите какой вывод можно сделать по этим данным.

In [ ]:
# реализация задачи с использованием SQL
pd.read_sql(
    """
    SELECT LSTAT, MEDV, RANK() OVER (ORDER BY LSTAT) as lstat_rank
    FROM (
      SELECT LSTAT, MEDV
      FROM boston_housing
      ORDER BY MEDV DESC
      LIMIT 25
)
ORDER BY lstat_rank
    """,
    con,
)

,LSTAT,MEDV,lstat_rank
0,1.73,50.0,1
1,1.92,50.0,2
2,2.88,50.0,3
3,2.96,50.0,4
4,2.97,50.0,5
5,3.01,46.0,6
6,3.11,44.0,7
7,3.16,50.0,8
8,3.26,50.0,9
9,3.32,50.0,10


In [ ]:
# реализация задачи с использованием библиотеки pandas
result_df = (df.nlargest(25, 'MEDV')[['LSTAT', 'MEDV']]
             .assign(lstat_rank=lambda x: x['LSTAT'].rank(method='min'))
             .sort_values('lstat_rank'))

print(result_df)

correlation = df['LSTAT'].corr(df['MEDV'])
print(f'Корреляция LSTAT-MEDV: {correlation}')

     LSTAT  MEDV  lstat_rank
161   1.73  50.0         1.0
162   1.92  50.0         2.0
204   2.88  50.0         3.0
370   2.96  50.0         4.0
195   2.97  50.0         5.0
282   3.01  46.0         6.0
256   3.11  44.0         7.0
283   3.16  50.0         8.0
368   3.26  50.0         9.0
163   3.32  50.0        10.0
98    3.57  43.8        11.0
166   3.70  50.0        12.0
369   3.73  50.0        13.0
280   3.76  45.4        14.0
203   3.81  48.5        15.0
228   3.92  46.7        16.0
233   3.95  48.3        17.0
224   4.14  44.8        18.0
186   4.45  50.0        19.0
225   4.63  50.0        20.0
257   5.12  50.0        21.0
262   5.91  48.8        22.0
267   7.44  50.0        23.0
372   8.88  50.0        24.0
371   9.53  50.0        25.0
Корреляция LSTAT-MEDV: -0.7376627261740151


В SQL сначала выбираются 25 самых дорогих домов, для них с помощью оконной функции вычисляются ранги для этой выборки. В python также сначала выбираются 25 дорогих домов и добавляется колонка с рангами от 1 до 25, метод min означает, что одинаковым значениям lstat - одинаковое значение ранга. В районах с дорогими домами сохраняется достаточно низкий процент населения с низким статусом от 1,73% до 9,53%. Корреляция показывает сильную обратную связь, то есть чем выше процент населения с низкими социально-экономическим статусом, тем дешевле будет стоить жилье в этом районе.

# 6. Выведите среднюю стоимость домов граничащих с рекой(CHAS) и нет (граничит/не граничит, стоимость)

In [ ]:
# реализация задачи с использованием SQL
pd.read_sql(
    """
    SELECT CHAS, ROUND(AVG(MEDV),2) as avg_cost, COUNT(*) as num_houses
    FROM boston_housing
    GROUP BY CHAS
    """,
    con,
)

,CHAS,avg_cost,num_houses
0,0,22.09,471
1,1,28.44,35


In [ ]:
# реализация задачи с использованием библиотеки pandas
result = (df.groupby('CHAS')['MEDV']
          .agg(avg_cost='mean', num_houses='count')
          .reset_index()
          .assign(location=lambda x: x['CHAS'].map({0: 'Не у реки', 1: 'У реки'}))
          [['location', 'avg_cost', 'num_houses']])
print(result.round(2))

    location  avg_cost  num_houses
0  Не у реки     22.09         471
1     У реки     28.44          35


В sql варианте было найдено среднее значение и количество домов и через группировку выведены результаты, где 0 - не у реки, 1 - у реки. В python также с помощью groupby делается группировка по CHAS, применяются агрегатные функции: среднее и подсчет общего количества. Как итог, дома у реки стоят дороже на $6,35 тысяч , при этом их значительно меньше - всего 6,9% от общей выборки, поэтому недвижимость у реки - достаточно эксклюзивная и дорогая.

# 7. Выведите все колонки, у которых среднее значение выше, когда дом граничит с рекой (название колонки) выбирая из CRIM, ZN, INDUS, CHAS, NOX. Напишите какой вывод можно сделать по этим данным.

In [ ]:
# реализация задачи с использованием SQL
pd.read_sql(
    """
    WITH river AS (
      SELECT
            AVG(CASE WHEN CHAS = 1 THEN CRIM ELSE NULL END) as crim_river,
            AVG(CASE WHEN CHAS = 0 THEN CRIM ELSE NULL END) as crim_no_river,
            AVG(CASE WHEN CHAS = 1 THEN ZN ELSE NULL END) as zn_river,
            AVG(CASE WHEN CHAS = 0 THEN ZN ELSE NULL END) as zn_no_river,
            AVG(CASE WHEN CHAS = 1 THEN INDUS ELSE NULL END) as indus_river,
            AVG(CASE WHEN CHAS = 0 THEN INDUS ELSE NULL END) as indus_no_river,
            AVG(CASE WHEN CHAS = 1 THEN NOX ELSE NULL END) as nox_river,
            AVG(CASE WHEN CHAS = 0 THEN NOX ELSE NULL END) as nox_no_river
        FROM boston_housing
    )
    SELECT "CRIM" as column, crim_river as avg_river, crim_no_river as avg_no_river, crim_river > crim_no_river as is_higher
    FROM river
    UNION ALL
    SELECT "ZN" as column, zn_river as avg_river, zn_no_river as avg_no_river, zn_river > zn_no_river as is_higher
    FROM river
    UNION ALL
    SELECT "INDUS" as column, indus_river as avg_river, indus_no_river as avg_no_river, indus_river > indus_no_river as is_higher
    FROM river
    UNION ALL
    SELECT "NOX" as column, nox_river as avg_river, nox_no_river as avg_no_river, nox_river > nox_no_river as is_higher
    FROM river
    """,
    con,
)

,column,avg_river,avg_no_river,is_higher
0,CRIM,1.851670,3.744447,0
1,ZN,7.714286,11.634820,0
2,INDUS,12.719143,11.019193,1
3,NOX,0.593426,0.551817,1


In [ ]:
# реализация задачи с использованием библиотеки pandas
check = ['CRIM', 'ZN', 'INDUS', 'NOX']
results = []
for col in check:
    mean_river = df[df['CHAS'] == 1][col].mean()
    mean_no_river = df[df['CHAS'] == 0][col].mean()
    is_higher = mean_river > mean_no_river

    results.append({
        'column': col,
        'avg_river': mean_river,
        'avg_no_river': mean_no_river,
        'is_higher': is_higher
    })
comparison_df = pd.DataFrame(results)
print(comparison_df)

  column  avg_river  avg_no_river  is_higher
0   CRIM   1.851670      3.744447      False
1     ZN   7.714286     11.634820      False
2  INDUS  12.719143     11.019193       True
3    NOX   0.593426      0.551817       True


В SQL была создана временная таблица со средними значениями, а с помощью условного оператора case были выведены группы по переменной chas, где 0 - не у реки, 1 - у реки, используя UNION ALL объединяются запросы. В Python фильтруются дома у реки и не у реки и вычисляется среднее, далее сравниваются значения и выводится логический результат. Как итог, среднее значение выше у реки в промышленных зонах и там, где есть концентрация азота, это означает, что у районов около реки есть ряд проблем с загрязнениями, выбросами и экологием в целом. При этом в домах около реки уровень преступности ниже и более плотная застройка. Таким образом, дома около реки противоречивы, так как с одной стороны там меньший уровень преступности и больше людей с высоким статусом, но находятся промышленные зоны и высокая концентрация азота.

# 8. Выведите значения долей промышленной застройки(INDUS), концентрации оксидов азота(NOX) и по их перцентилям - 10, 20 ... 100 ( перцетиль(10,20...100),значение INDUS, значение NOX). Напишите прослеживается ли между ними взаимосвязь

In [ ]:
# реализация задачи с использованием SQL
pd.read_sql(
    """
    WITH percentiles AS (
        SELECT
            10 as percentile,
            (SELECT INDUS FROM boston_housing ORDER BY INDUS LIMIT 1 OFFSET (SELECT COUNT(*) / 10 FROM boston_housing)) as indus_value,
            (SELECT NOX FROM boston_housing ORDER BY NOX LIMIT 1 OFFSET (SELECT COUNT(*) / 10 FROM boston_housing)) as nox_value
        UNION ALL
        SELECT
            20 as percentile,
            (SELECT INDUS FROM boston_housing ORDER BY INDUS LIMIT 1 OFFSET (SELECT COUNT(*) / 5 FROM boston_housing)) as indus_value,
            (SELECT NOX FROM boston_housing ORDER BY NOX LIMIT 1 OFFSET (SELECT COUNT(*) / 5 FROM boston_housing)) as nox_value
        UNION ALL
        SELECT
            30 as percentile,
            (SELECT INDUS FROM boston_housing ORDER BY INDUS LIMIT 1 OFFSET (SELECT COUNT(*) * 3 / 10 FROM boston_housing)) as indus_value,
            (SELECT NOX FROM boston_housing ORDER BY NOX LIMIT 1 OFFSET (SELECT COUNT(*) * 3 / 10 FROM boston_housing)) as nox_value
        UNION ALL
        SELECT
            40 as percentile,
            (SELECT INDUS FROM boston_housing ORDER BY INDUS LIMIT 1 OFFSET (SELECT COUNT(*) * 2 / 5 FROM boston_housing)) as indus_value,
            (SELECT NOX FROM boston_housing ORDER BY NOX LIMIT 1 OFFSET (SELECT COUNT(*) * 2 / 5 FROM boston_housing)) as nox_value
        UNION ALL
        SELECT
            50 as percentile,
            (SELECT INDUS FROM boston_housing ORDER BY INDUS LIMIT 1 OFFSET (SELECT COUNT(*) / 2 FROM boston_housing)) as indus_value,
            (SELECT NOX FROM boston_housing ORDER BY NOX LIMIT 1 OFFSET (SELECT COUNT(*) / 2 FROM boston_housing)) as nox_value
        UNION ALL
        SELECT
            60 as percentile,
            (SELECT INDUS FROM boston_housing ORDER BY INDUS LIMIT 1 OFFSET (SELECT COUNT(*) * 3 / 5 FROM boston_housing)) as indus_value,
            (SELECT NOX FROM boston_housing ORDER BY NOX LIMIT 1 OFFSET (SELECT COUNT(*) * 3 / 5 FROM boston_housing)) as nox_value
        UNION ALL
        SELECT
            70 as percentile,
            (SELECT INDUS FROM boston_housing ORDER BY INDUS LIMIT 1 OFFSET (SELECT COUNT(*) * 7 / 10 FROM boston_housing)) as indus_value,
            (SELECT NOX FROM boston_housing ORDER BY NOX LIMIT 1 OFFSET (SELECT COUNT(*) * 7 / 10 FROM boston_housing)) as nox_value
        UNION ALL
        SELECT
            80 as percentile,
            (SELECT INDUS FROM boston_housing ORDER BY INDUS LIMIT 1 OFFSET (SELECT COUNT(*) * 4 / 5 FROM boston_housing)) as indus_value,
            (SELECT NOX FROM boston_housing ORDER BY NOX LIMIT 1 OFFSET (SELECT COUNT(*) * 4 / 5 FROM boston_housing)) as nox_value
        UNION ALL
        SELECT
            90 as percentile,
            (SELECT INDUS FROM boston_housing ORDER BY INDUS LIMIT 1 OFFSET (SELECT COUNT(*) * 9 / 10 FROM boston_housing)) as indus_value,
            (SELECT NOX FROM boston_housing ORDER BY NOX LIMIT 1 OFFSET (SELECT COUNT(*) * 9 / 10 FROM boston_housing)) as nox_value
        UNION ALL
        SELECT
            100 as percentile,
            (SELECT MAX(INDUS) FROM boston_housing) as indus_value,
            (SELECT MAX(NOX) FROM boston_housing) as nox_value
    )
    SELECT * FROM percentiles

    """,
    con,
)

,percentile,indus_value,nox_value
0,10,2.89,0.426
1,20,4.39,0.442
2,30,5.96,0.472
3,40,7.38,0.507
4,50,9.69,0.538
5,60,12.83,0.575
6,70,18.10,0.605
7,80,18.10,0.668
8,90,19.58,0.713
9,100,27.74,0.871


In [ ]:
# реализация задачи с использованием библиотеки pandas
percentiles = range(10, 101, 10)
result_df = pd.DataFrame([(p, df['INDUS'].quantile(p/100), df['NOX'].quantile(p/100))
                         for p in percentiles],
                        columns=['percentile', 'INDUS', 'NOX'])
print(result_df)

correlation = df['INDUS'].corr(df['NOX'])
print(f'Корреляция между INDUS-NOX: {correlation.round(3)}')


   percentile  INDUS    NOX
0          10   2.91  0.427
1          20   4.39  0.442
2          30   5.96  0.472
3          40   7.38  0.507
4          50   9.69  0.538
5          60  12.83  0.575
6          70  18.10  0.605
7          80  18.10  0.668
8          90  19.58  0.713
9         100  27.74  0.871
Корреляция между INDUS-NOX: 0.764


Чтобы найти перцентили значения долей промышленной зоны, значения были отсортированы по возрастанию, затем вычисляются 10% от общего количества строк, затем offset пропускает первые 10% строк. В pandas есть встроенная функция перцентелей quantile(p/100), с ее помощью вычисляются значения. Если говорить про результаты, то в наименее промышленных районах загрязнение небольшое, пр  переходе к средним значениям промышленности, растет и загрязнение и в наиболее промышленны районах наблюдается максимальное загрязнение. Корреляция, которая составляет 76% подтверждает прямую связь между переменными: при увеличении значения доли промышленной застройки растет и уровень загрязнения.